# 🎯 Objetivo

### Visualizar los resultados de detección de anomalías de forma interactiva y clara, de modo que:

- Se puedan inspeccionar runs por pozo y etapa.
- Se vean los scores de anomalía en profundidad.
- Permita un primer paso hacia una app operativa o dashboard en producción.

### Visualización interactiva en Jupyter usando Plotly con dropdowns.

Enfocado en:

- Mostrar la señal CCL y su score de anomalía en profundidad.
- Permitir elegir un pozo y etapa con controles interactivos.
- Resaltar visualmente las zonas con alta probabilidad de anomalía.

### 💻 Celda 2 - Carga de datos con resultados de anomalía

In [1]:
import pandas as pd

# Dataset procesado en etapa anterior
df = pd.read_csv(r"C:\Developer\fundamentos\data\ccl_anomaly_scores.csv")

# Vista rápida
df.head()

,DEPT,CCL,TENS,archivo_origen,pozo,sentido,etapa,CCL_norm,dCCL,abs_dCCL,...,abs_dCCL_max,TENS_mean,TENS_std,TENS_max,anomaly_iso,score_iso,anomaly_svm,anomaly_lof,score_knn,anomaly_knn
0,2800.05,-0.00004,1732.00004,BPE-2343_E48_Up__27Nov24_130708.las,BPE-2343,Up,E48,-0.008006,NaN,NaN,...,19.481585,1583.567427,158.066178,2023.00015,1,-0.158595,1,1,0.392406,0
1,2800.10,-0.00212,1732.00004,BPE-2343_E48_Up__27Nov24_130708.las,BPE-2343,Up,E48,-0.424339,-0.416333,0.416333,...,19.481585,1583.567427,158.066178,2023.00015,1,-0.143797,1,1,0.514461,0
2,2800.15,0.00143,1669.99998,BPE-2343_E48_Up__27Nov24_130708.las,BPE-2343,Up,E48,0.286229,0.710568,0.710568,...,19.481585,1583.567427,158.066178,2023.00015,1,-0.168060,1,1,0.422810,0
3,2800.20,0.00558,1669.99998,BPE-2343_E48_Up__27Nov24_130708.las,BPE-2343,Up,E48,1.116894,0.830665,0.830665,...,19.481585,1583.567427,158.066178,2023.00015,1,-0.181208,1,1,0.426876,0
4,2800.25,0.00775,1669.99998,BPE-2343_E48_Up__27Nov24_130708.las,BPE-2343,Up,E48,1.551241,0.434347,0.434347,...,19.481585,1583.567427,158.066178,2023.00015,1,-0.184355,1,1,0.520572,0


### 🧰 Celda 3 - Herramientas interactivas

In [2]:
import plotly.graph_objs as go
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

### 📋 Celda 4 - Selección de pozo y etapa

In [3]:
# Widgets para selección interactiva
pozos = sorted(df["pozo"].dropna().unique())
selector_pozo = widgets.Dropdown(options=pozos, description="Pozo")

def update_etapas(pozo_seleccionado):
    etapas = df[df["pozo"] == pozo_seleccionado]["etapa"].dropna().unique()
    return sorted(etapas)

selector_etapa = widgets.Dropdown(description="Etapa")

def on_pozo_change(change):
    selector_etapa.options = update_etapas(change['new'])

selector_pozo.observe(on_pozo_change, names='value')
on_pozo_change({'new': selector_pozo.value})

display(selector_pozo, selector_etapa)

Dropdown(description='Pozo', options=('BPE-2343',), value='BPE-2343')

Dropdown(description='Etapa', options=('E36', 'E38', 'E46', 'E48'), value=None)

### 📈 Celda 5 - Función para graficar run

In [20]:
def plot_etapa(df, pozo, etapa):
    df_etapa = df[(df["pozo"] == pozo) & (df["etapa"] == etapa)].sort_values(by="DEPT").copy()

    fig = go.Figure()

    # Línea base: señal CCL
    fig.add_trace(go.Scatter(
        x=df_etapa["CCL"], 
        y=df_etapa["DEPT"], 
        mode='lines',
        name='CCL',
        line=dict(color='blue')
    ))

    # Promedios por tramo (25m y 50m)
    for step, color, dash in [(15, "orange", "dot"), (30, "black", "solid")]:
        df_etapa[f"DEPT_bin_{step}"] = (df_etapa["DEPT"] // step) * step
        grouped = df_etapa.groupby(f"DEPT_bin_{step}")["score_iso"].mean().reset_index()

        fig.add_trace(go.Scatter(
            x=grouped["score_iso"],
            y=grouped[f"DEPT_bin_{step}"],
            mode='lines+markers',
            name=f"Score medio cada {step}m",
            line=dict(color=color, dash=dash, width=2),
            marker=dict(symbol="circle", size=6)
        ))

    # Configuración visual
    fig.update_layout(
        title=f"Pozo: {pozo} | Etapa: {etapa}",
        xaxis_title="Valor",
        yaxis_title="Profundidad (DEPT)",
        yaxis_autorange='reversed',
        height=600,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=20, r=20, t=40, b=20),
    )

    fig.show()


### 🖼️ Celda 6 - Ejecutar visualización

#### 💡 Nota: Podés volver a ejecutar la celda 6 con diferentes selecciones en los dropdowns para ver otros runs.

In [21]:
plot_etapa(df, selector_pozo.value, selector_etapa.value)

In [6]:
def plot_etapa(df, pozo, etapa):
    df_etapa = df[(df["pozo"] == pozo) & (df["etapa"] == etapa)].sort_values(by="DEPT").copy()

    # ✅ Suavizado de CCL con media móvil
    df_etapa["CCL_smooth"] = df_etapa["CCL"].rolling(window=5, min_periods=1).mean()

    # 🔍 Score por tramo
    resumen_25 = df_etapa.copy()
    resumen_25["DEPT_bin_25"] = (resumen_25["DEPT"] // 25) * 25
    grouped_25 = resumen_25.groupby("DEPT_bin_25")["score_iso"].mean().reset_index().rename(columns={"score_iso": "score_mean_25"})

    resumen_50 = df_etapa.copy()
    resumen_50["DEPT_bin_50"] = (resumen_50["DEPT"] // 50) * 50
    grouped_50 = resumen_50.groupby("DEPT_bin_50")["score_iso"].mean().reset_index().rename(columns={"score_iso": "score_mean_50"})

    # 📈 Gráfico
    fig = go.Figure()

    # CCL suavizado
    fig.add_trace(go.Scatter(
        x=df_etapa["CCL_smooth"],
        y=df_etapa["DEPT"],
        mode='lines',
        name='CCL suavizado',
        line=dict(color='blue', width=2)
    ))

    # Score medio por 25m
    fig.add_trace(go.Scatter(
        x=grouped_25["score_mean_25"],
        y=grouped_25["DEPT_bin_25"],
        mode='lines+markers',
        name='Score medio cada 25m',
        line=dict(color='orange', dash='dot', width=2),
        marker=dict(size=6)
    ))

    # Score medio por 50m
    fig.add_trace(go.Scatter(
        x=grouped_50["score_mean_50"],
        y=grouped_50["DEPT_bin_50"],
        mode='lines+markers',
        name='Score medio cada 50m',
        line=dict(color='black', width=2),
        marker=dict(size=6)
    ))

    # Configuración visual
    fig.update_layout(
        title=f"Pozo: {pozo} | Etapa: {etapa}",
        xaxis_title="Valor",
        yaxis_title="Profundidad (DEPT)",
        yaxis_autorange='reversed',
        height=650,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=20, r=20, t=40, b=20),
    )

    fig.show()

    # 🧾 Tabla de tramos con mayor score (top 10)
    print(f"\n📋 Top tramos por promedio de score (25m):")
    display(grouped_25.sort_values("score_mean_25", ascending=False).head(10))

    print(f"\n📋 Top tramos por promedio de score (50m):")
    display(grouped_50.sort_values("score_mean_50", ascending=False).head(10))


In [8]:
import plotly.graph_objs as go
import plotly.colors
import pandas as pd

def plot_etapa(df, pozo, etapa):
    df_etapa = df[(df["pozo"] == pozo) & (df["etapa"] == etapa)].sort_values(by="DEPT").copy()
    df_etapa["CCL_smooth"] = df_etapa["CCL"].rolling(window=5, min_periods=1).mean()

    # Tramos de 25m con promedio de score
    df_etapa["DEPT_bin_25"] = (df_etapa["DEPT"] // 25) * 25
    grouped_25 = df_etapa.groupby("DEPT_bin_25")["score_iso"].mean().reset_index().rename(columns={"score_iso": "score_mean_25"})

    # Top 5 zonas más anómalas
    top5 = grouped_25.sort_values("score_mean_25", ascending=False).head(5).reset_index(drop=True)

    # Escalamos los scores para usar en opacidad o color
    max_score = top5["score_mean_25"].max()
    min_score = top5["score_mean_25"].min()
    score_range = max_score - min_score + 1e-5

    # Colores de calor: rojo más fuerte = más riesgo
    colorscale = plotly.colors.sequential.OrRd

    def score_to_color(score):
        norm = (score - min_score) / score_range
        idx = int(norm * (len(colorscale) - 1))
        return colorscale[idx]

    # Iniciar gráfico
    fig = go.Figure()

    # Curva CCL suavizada
    fig.add_trace(go.Scatter(
        x=df_etapa["CCL_smooth"],
        y=df_etapa["DEPT"],
        mode='lines',
        name='CCL suavizado',
        line=dict(color='blue', width=2)
    ))

    # Score medio por 25m
    fig.add_trace(go.Scatter(
        x=grouped_25["score_mean_25"],
        y=grouped_25["DEPT_bin_25"],
        mode='lines+markers',
        name='Score medio cada 25m',
        line=dict(color='orange', dash='dot', width=2),
        marker=dict(size=6)
    ))

    # Agregar franjas de riesgo
    for i, row in top5.iterrows():
        y0 = row["DEPT_bin_25"]
        y1 = y0 + 25
        color = score_to_color(row["score_mean_25"])
        fig.add_shape(
            type="rect",
            x0=0, x1=1,  # abarca todo el eje X
            y0=y0, y1=y1,
            xref="paper", yref="y",
            fillcolor=color,
            opacity=0.3,
            line_width=0,
            layer="below"
        )

    # Layout
    fig.update_layout(
        title=f"Pozo: {pozo} | Etapa: {etapa}",
        xaxis_title="Valor",
        yaxis_title="Profundidad (DEPT)",
        yaxis_autorange='reversed',
        height=650,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=20, r=20, t=40, b=20),
    )

    fig.show()

    # Mostrar tabla de riesgo
    print("📋 Top 5 zonas de riesgo (25m):")
    display(top5)


In [10]:
import plotly.graph_objs as go
import plotly.colors
import pandas as pd

def plot_etapa(df, pozo, etapa):
    df_etapa = df[(df["pozo"] == pozo) & (df["etapa"] == etapa)].sort_values(by="DEPT").copy()
    df_etapa["CCL_smooth"] = df_etapa["CCL"].rolling(window=5, min_periods=1).mean()

    # Calcular score promedio cada 50m
    df_etapa["DEPT_bin_50"] = (df_etapa["DEPT"] // 50) * 50
    grouped_50 = df_etapa.groupby("DEPT_bin_50")["score_iso"].mean().reset_index().rename(columns={"score_iso": "score_mean_50"})

    # Top 5 zonas más anómalas (50m)
    top5 = grouped_50.sort_values("score_mean_50", ascending=False).head(5).reset_index(drop=True)

    # Escalamos los scores para color
    max_score = top5["score_mean_50"].max()
    min_score = top5["score_mean_50"].min()
    score_range = max_score - min_score + 1e-5
    colorscale = plotly.colors.sequential.OrRd

    def score_to_color(score):
        norm = (score - min_score) / score_range
        idx = int(norm * (len(colorscale) - 1))
        return colorscale[idx]

    # Iniciar gráfico
    fig = go.Figure()

    # CCL suavizado
    fig.add_trace(go.Scatter(
        x=df_etapa["CCL_smooth"],
        y=df_etapa["DEPT"],
        mode='lines',
        name='CCL suavizado',
        line=dict(color='blue', width=2)
    ))

    # Score medio por 50m
    fig.add_trace(go.Scatter(
        x=grouped_50["score_mean_50"],
        y=grouped_50["DEPT_bin_50"],
        mode='lines+markers',
        name='Score medio cada 50m',
        line=dict(color='black', width=2),
        marker=dict(size=6)
    ))

    # Agregar franjas de riesgo en top 5 zonas
    for i, row in top5.iterrows():
        y0 = row["DEPT_bin_50"]
        y1 = y0 + 50
        color = score_to_color(row["score_mean_50"])
        fig.add_shape(
            type="rect",
            x0=0, x1=1,
            y0=y0, y1=y1,
            xref="paper", yref="y",
            fillcolor=color,
            opacity=0.3,
            line_width=0,
            layer="below"
        )

    # Layout final
    fig.update_layout(
        title=f"Pozo: {pozo} | Etapa: {etapa}",
        xaxis_title="Valor",
        yaxis_title="Profundidad (DEPT)",
        yaxis_autorange='reversed',
        height=650,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=20, r=20, t=40, b=20),
    )

    fig.show()

    # Tabla informativa
    print("📋 Top 5 zonas de riesgo (50m):")
    display(top5)


### 💬 Celda 7 - Conclusiones y próximos pasos

#### ✅ Conclusiones

- Esta visualización permite inspeccionar visualmente la señal de CCL y los scores de anomalía por etapa.
- Las zonas de alto score pueden interpretarse como candidatos a casing restriction.
- El enfoque se puede extender a otras métricas o modelos.

**Próximo paso sugerido**: Transformar esto en una app interactiva con `Streamlit` para facilitar el uso por parte del equipo de operaciones.
